LifeTwin AI — Data Preprocessing

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv(r"C:\Users\ABC\OneDrive - Aditya Educational Institutions\Desktop\LifeTwin\LifeTwin\LifeTwin\data\raw\lifetwin_(2) (1).csv")

print("Dataset shape:", df.shape)

Dataset shape: (10000, 20)


In [3]:
metadata_features = [
    "user_id",
    "record_date"
]

context_features = [
    "age_group",
    "day_of_week"
]

behavioral_features = [
    "sleep_hours",
    "sleep_quality",
    "wake_time_consistency",
    "study_work_hours",
    "focus_sessions",
    "screen_time_hours",
    "social_media_hours",
    "notification_interruptions",
    "exercise_minutes",
    "step_count",
    "stress_level",
    "mood_score",
    "energy_level",
    "caffeine_intake",
    "break_minutes"
]

target = "productivity_score"

In [4]:
print("Metadata:", metadata_features)
print("Context:", context_features)
print("Behavioral:", len(behavioral_features))
print("Target:", target)

Metadata: ['user_id', 'record_date']
Context: ['age_group', 'day_of_week']
Behavioral: 15
Target: productivity_score


## Missing Value Analysis

In [5]:
missing = df.isnull().sum()

missing_table = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": (missing / len(df)) * 100
})

missing_table = missing_table[
    missing_table["missing_count"] > 0
].sort_values("missing_count", ascending=False)

missing_table

,missing_count,missing_percent
step_count,494,4.94
mood_score,490,4.90
study_work_hours,487,4.87
wake_time_consistency,476,4.76
focus_sessions,469,4.69
break_minutes,453,4.53
exercise_minutes,414,4.14
sleep_hours,400,4.00
social_media_hours,395,3.95
screen_time_hours,364,3.64


In [6]:
missing_per_user = (
    df.groupby("user_id")[behavioral_features]
      .apply(lambda x: x.isnull().sum().sum())
)

print(missing_per_user.describe())
print("Users with at least one missing value:",
      (missing_per_user > 0).sum())

count    250.000000
mean      24.116000
std        5.095725
min       12.000000
25%       21.000000
50%       24.000000
75%       27.000000
max       41.000000
dtype: float64
Users with at least one missing value: 250


In [7]:
records_per_user = df["user_id"].value_counts()

print(records_per_user.describe())

count    250.0
mean      40.0
std        0.0
min       40.0
25%       40.0
50%       40.0
75%       40.0
max       40.0
Name: count, dtype: float64


In [8]:
print("Minimum records per user:", records_per_user.min())
print("Maximum records per user:", records_per_user.max())
print("Total users:", records_per_user.size)

Minimum records per user: 40
Maximum records per user: 40
Total users: 250


In [ ]:
from sklearn.model_selection import train_test_split

users = df["user_id"].astype(str).unique().tolist()

print("Total users:", len(users))


Total users: 250
Total users: 250


In [12]:
from sklearn.model_selection import train_test_split

train_users, temp_users = train_test_split(
    users,
    test_size=0.30,
    random_state=42
)

val_users, test_users = train_test_split(
    temp_users,
    test_size=0.50,
    random_state=42
)

print("Training users:", len(train_users))
print("Validation users:", len(val_users))
print("Test users:", len(test_users))

Training users: 175
Validation users: 37
Test users: 38


In [13]:
train_df = df[df["user_id"].isin(train_users)].copy()
val_df = df[df["user_id"].isin(val_users)].copy()
test_df = df[df["user_id"].isin(test_users)].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (7000, 20)
Validation shape: (1480, 20)
Test shape: (1520, 20)


In [14]:
print(
    "Users overlap:",
    len(set(train_users) & set(val_users)),
    len(set(train_users) & set(test_users)),
    len(set(val_users) & set(test_users))
)

Users overlap: 0 0 0


In [15]:
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (7000, 20)
Validation shape: (1480, 20)
Test shape: (1520, 20)


In [16]:
print(
    "Users overlap:",
    len(set(train_users) & set(val_users)),
    len(set(train_users) & set(test_users)),
    len(set(val_users) & set(test_users))
)

Users overlap: 0 0 0


Handle missing values

In [17]:
from sklearn.impute import SimpleImputer

In [18]:
imputer = SimpleImputer(strategy="median")

In [19]:
imputer.fit(train_df[behavioral_features])

,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feature has nomissing values at fit/train time, the feature won't appear onthe missing indicator even if there are missing values attransform/test time.",False
,"keep_empty_features keep_empty_features: bool, default=FalseIf True, features that consist exclusively of missing values when`fit` is called are returned in results when `transform` is called.The imputed value is always `0` except when `strategy=""constant""`in which case `fill_value` will be used instead... versionadded:: 1.2",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](15,)","['sleep_hours','sleep_quality','wake_time_consistency',...,'energy_level', 'caffeine_intake','break_minutes']"
indicator_ indicator_: :class:`~sklearn.impute.MissingIndicator`Indicator used to add binary indicators for missing values.`None` if `add_indicator=False`.,NoneType,None
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,15
"statistics_ statistics_: array of shape (n_features,)The imputation fill value for each feature.Computing statistics can result in `np.nan` values.During :meth:`transform`, features corresponding to `np.nan`statistics will be discarded.","ndarray[float64](15,)","[ 6.76, 5.7 , 5.4 ,..., 5.5 , 2.2 ,57. ]"


In [20]:
train_medians = pd.Series(
    imputer.statistics_,
    index=behavioral_features
)

train_medians

sleep_hours                      6.76
sleep_quality                    5.70
wake_time_consistency            5.40
study_work_hours                 5.66
focus_sessions                   3.60
screen_time_hours                6.84
social_media_hours               1.94
notification_interruptions      40.00
exercise_minutes                26.00
step_count                    5670.00
stress_level                     5.20
mood_score                       5.80
energy_level                     5.50
caffeine_intake                  2.20
break_minutes                   57.00
dtype: float64

In [21]:
train_df[behavioral_features] = imputer.transform(
    train_df[behavioral_features]
)

val_df[behavioral_features] = imputer.transform(
    val_df[behavioral_features]
)

test_df[behavioral_features] = imputer.transform(
    test_df[behavioral_features]
)

In [22]:
print("Train missing values:")
print(train_df[behavioral_features].isnull().sum().sum())

print("\nValidation missing values:")
print(val_df[behavioral_features].isnull().sum().sum())

print("\nTest missing values:")
print(test_df[behavioral_features].isnull().sum().sum())

Train missing values:
0

Validation missing values:
0

Test missing values:
0


categorical + date preprocessing

In [23]:
print("Target missing values:")
print(train_df[target].isnull().sum())
print(val_df[target].isnull().sum())
print(test_df[target].isnull().sum())

Target missing values:
0
0
0


In [24]:
print("Target statistics after preprocessing:")

print("Train:")
print(train_df[target].describe())

print("\nValidation:")
print(val_df[target].describe())

print("\nTest:")
print(test_df[target].describe())

Target statistics after preprocessing:
Train:
count    7000.000000
mean       48.810371
std        18.102503
min         0.000000
25%        35.100000
50%        49.800000
75%        62.800000
max        93.400000
Name: productivity_score, dtype: float64

Validation:
count    1480.000000
mean       47.258176
std        17.737393
min         0.000000
25%        34.575000
50%        46.800000
75%        60.700000
max        94.300000
Name: productivity_score, dtype: float64

Test:
count    1520.000000
mean       44.231579
std        18.738049
min         0.000000
25%        29.800000
50%        43.400000
75%        59.225000
max        93.300000
Name: productivity_score, dtype: float64


In [25]:
print("Minimum date:", df["record_date"].min())
print("Maximum date:", df["record_date"].max())

Minimum date: 2026-02-01
Maximum date: 2026-07-20


In [26]:
print(df["record_date"].head())

0    2026-05-13
1    2026-07-14
2    2026-03-29
3    2026-04-10
4    2026-05-21
Name: record_date, dtype: str


In [27]:
for dataset in [train_df, val_df, test_df]:
    dataset["record_date"] = pd.to_datetime(
        dataset["record_date"],
        errors="coerce"
    )

In [28]:
print(train_df["record_date"].dtype)
print(val_df["record_date"].dtype)
print(test_df["record_date"].dtype)

datetime64[us]
datetime64[us]
datetime64[us]


In [29]:
print("Age groups:")
print(train_df["age_group"].value_counts())

print("\nDays:")
print(train_df["day_of_week"].value_counts())

Age groups:
age_group
18-22    3840
23-27    2160
28-35    1000
Name: count, dtype: int64

Days:
day_of_week
Sunday       1042
Saturday     1022
Thursday     1003
Wednesday     995
Monday        984
Friday        980
Tuesday       974
Name: count, dtype: int64


In [30]:
print("Validation age groups:")
print(val_df["age_group"].value_counts())

print("\nTest age groups:")
print(test_df["age_group"].value_counts())

Validation age groups:
age_group
18-22    680
23-27    560
28-35    240
Name: count, dtype: int64

Test age groups:
age_group
18-22    800
23-27    600
28-35    120
Name: count, dtype: int64


In [31]:
print("Validation days:")
print(val_df["day_of_week"].value_counts())

print("\nTest days:")
print(test_df["day_of_week"].value_counts())

Validation days:
day_of_week
Monday       231
Saturday     214
Friday       213
Sunday       212
Thursday     210
Wednesday    201
Tuesday      199
Name: count, dtype: int64

Test days:
day_of_week
Friday       228
Monday       221
Saturday     221
Tuesday      220
Sunday       218
Thursday     210
Wednesday    202
Name: count, dtype: int64


## Categorical Encoding

In [32]:
from sklearn.preprocessing import OneHotEncoder

categorical_features = ["age_group", "day_of_week"]

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

encoder.fit(train_df[categorical_features])

train_encoded = encoder.transform(train_df[categorical_features])
val_encoded = encoder.transform(val_df[categorical_features])
test_encoded = encoder.transform(test_df[categorical_features])

encoded_columns = encoder.get_feature_names_out(categorical_features)

train_encoded_df = pd.DataFrame(
    train_encoded,
    columns=encoded_columns,
    index=train_df.index
)

val_encoded_df = pd.DataFrame(
    val_encoded,
    columns=encoded_columns,
    index=val_df.index
)

test_encoded_df = pd.DataFrame(
    test_encoded,
    columns=encoded_columns,
    index=test_df.index
)

print("Encoded features:")
print(list(encoded_columns))

print("\nTrain shape:", train_encoded_df.shape)
print("Validation shape:", val_encoded_df.shape)
print("Test shape:", test_encoded_df.shape)

Encoded features:
['age_group_18-22', 'age_group_23-27', 'age_group_28-35', 'day_of_week_Friday', 'day_of_week_Monday', 'day_of_week_Saturday', 'day_of_week_Sunday', 'day_of_week_Thursday', 'day_of_week_Tuesday', 'day_of_week_Wednesday']

Train shape: (7000, 10)
Validation shape: (1480, 10)
Test shape: (1520, 10)


In [33]:
print(train_encoded_df.head())
print("\nUnique values in encoded data:")
print(train_encoded_df.nunique())

   age_group_18-22  age_group_23-27  age_group_28-35  day_of_week_Friday  \
0              0.0              1.0              0.0                 0.0   
1              1.0              0.0              0.0                 0.0   
2              0.0              1.0              0.0                 0.0   
3              0.0              1.0              0.0                 1.0   
4              1.0              0.0              0.0                 0.0   

   day_of_week_Monday  day_of_week_Saturday  day_of_week_Sunday  \
0                 0.0                   0.0                 0.0   
1                 0.0                   0.0                 0.0   
2                 0.0                   0.0                 1.0   
3                 0.0                   0.0                 0.0   
4                 0.0                   0.0                 0.0   

   day_of_week_Thursday  day_of_week_Tuesday  day_of_week_Wednesday  
0                   0.0                  0.0                    1.0  


## Behavioral Feature Scaling

In [34]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit only on training behavioral features
scaler.fit(train_df[behavioral_features])

# Transform train, validation and test
train_behavior_scaled = scaler.transform(train_df[behavioral_features])
val_behavior_scaled = scaler.transform(val_df[behavioral_features])
test_behavior_scaled = scaler.transform(test_df[behavioral_features])

# Convert to DataFrames
train_behavior_scaled_df = pd.DataFrame(
    train_behavior_scaled,
    columns=behavioral_features,
    index=train_df.index
)

val_behavior_scaled_df = pd.DataFrame(
    val_behavior_scaled,
    columns=behavioral_features,
    index=val_df.index
)

test_behavior_scaled_df = pd.DataFrame(
    test_behavior_scaled,
    columns=behavioral_features,
    index=test_df.index
)

print("Scaled train shape:", train_behavior_scaled_df.shape)
print("Scaled validation shape:", val_behavior_scaled_df.shape)
print("Scaled test shape:", test_behavior_scaled_df.shape)

Scaled train shape: (7000, 15)
Scaled validation shape: (1480, 15)
Scaled test shape: (1520, 15)


In [35]:
print("Training means after scaling:")
print(train_behavior_scaled_df.mean().round(4))

print("\nTraining standard deviations after scaling:")
print(train_behavior_scaled_df.std().round(4))

Training means after scaling:
sleep_hours                  -0.0
sleep_quality                 0.0
wake_time_consistency        -0.0
study_work_hours             -0.0
focus_sessions               -0.0
screen_time_hours            -0.0
social_media_hours            0.0
notification_interruptions    0.0
exercise_minutes             -0.0
step_count                    0.0
stress_level                 -0.0
mood_score                    0.0
energy_level                  0.0
caffeine_intake              -0.0
break_minutes                -0.0
dtype: float64

Training standard deviations after scaling:
sleep_hours                   1.0001
sleep_quality                 1.0001
wake_time_consistency         1.0001
study_work_hours              1.0001
focus_sessions                1.0001
screen_time_hours             1.0001
social_media_hours            1.0001
notification_interruptions    1.0001
exercise_minutes              1.0001
step_count                    1.0001
stress_level                  

## Save Processed Data

In [36]:
# Create clean copies
train_clean = train_df.copy()
val_clean = val_df.copy()
test_clean = test_df.copy()

print("Train:", train_clean.shape)
print("Validation:", val_clean.shape)
print("Test:", test_clean.shape)

Train: (7000, 20)
Validation: (1480, 20)
Test: (1520, 20)


In [37]:
processed_df = pd.concat(
    [train_clean, val_clean, test_clean],
    axis=0
).sort_index()

processed_df.to_csv(
    "../data/processed/lifetwin_clean_dataset.csv",
    index=False
)

print("Processed dataset saved.")
print("Shape:", processed_df.shape)

Processed dataset saved.
Shape: (10000, 20)


In [40]:
import joblib

joblib.dump(
    imputer,
    "../models/preprocessing/imputer.pkl"
)

joblib.dump(
    encoder,
    "../models/preprocessing/encoder.pkl"
)

joblib.dump(
    scaler,
    "../models/preprocessing/scaler.pkl"
)

print("Preprocessing artifacts saved successfully.")

Preprocessing artifacts saved successfully.


In [41]:
import os

files = [
    "../data/processed/lifetwin_clean_dataset.csv",
    "../models/preprocessing/imputer.pkl",
    "../models/preprocessing/encoder.pkl",
    "../models/preprocessing/scaler.pkl"
]

for file in files:
    print(file, "->", os.path.exists(file))

../data/processed/lifetwin_clean_dataset.csv -> True
../models/preprocessing/imputer.pkl -> True
../models/preprocessing/encoder.pkl -> True
../models/preprocessing/scaler.pkl -> True
